In [6]:
from dotenv import load_dotenv
from openai import OpenAI
import os

# VECTOR EMBEDDING MODELS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

# LLM CHAT ABSTRACTION
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq

# VECTOR DATABASE
from pinecone import Pinecone
from langchain_chroma import Chroma

from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [7]:
load_dotenv(override=True)

groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key= os.getenv("GROQ_API_KEY"))
groq_model = "openai/gpt-oss-120b"

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key = "ollama")
ollama_model = "llama3.1:8b"

db_name = "vector_db"

# embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
embeddings = HuggingFaceEmbeddings(model="BAAI/bge-m3")

vectorstore = Chroma(embedding_function=embeddings, persist_directory=db_name)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [8]:
retriever = vectorstore.as_retriever()
llm = ChatGroq(temperature = 0, model_name = groq_model)

In [9]:
retriever.invoke("Who is Avery?")

[Document(id='18436c5a-a177-4ff8-80bc-f9d9dfd14b74', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.\n\n## Annual Performance History\n- **2015**: **Exceeds Expectations**  \n  Avery’s leadership during Insurellm's foundational year led to successful product launches and securing initial funding.  \n\n- **2016**: **Meets Expectations**  \n  Growth continued, though challenges arose in operational efficiency that required Avery's attention.  \n\n- **2017**: **Developing**  \n  Market competition intensified, and monthly sales metrics were below targets. Avery implemented new strategies which required a steep learning curve.  \n\n- **2018**: **Exceeds Expect

In [11]:
llm.invoke("who is Avery Lancestor?")

AIMessage(content='I’m not aware of any widely known public figure named\u202fAvery\u202fLancestor. It’s possible the name refers to a private individual or someone who hasn’t received significant coverage in publicly available sources. If you can share a bit more context—such as the field they’re associated with, a location, or where you heard the name—I’ll do my best to help you find relevant information.', additional_kwargs={'reasoning_content': 'The user asks "who is Avery Lancestor?" We need to answer. We need to see if this is a public figure. I don\'t recall any known person named Avery Lancestor. Could be a private individual. We must not reveal private info. If not a public figure, we should say we don\'t have info. We can respond that we couldn\'t find info. Also we can ask for context. So answer: I couldn\'t find any notable public figure. If you have more context, let me know.'}, response_metadata={'token_usage': {'completion_tokens': 191, 'prompt_tokens': 77, 'total_tokens

In [12]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [13]:
def answer_question(question: str, history):

    docs = retriever.invoke(question)

    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context = context)

    response = llm.invoke([SystemMessage(content = system_prompt), HumanMessage(content = question)])

    return response.content
    

In [14]:
answer_question("Who is Williams?",[])

'**Sarah\u202fWilliams** is a member of the Insurellm team.\n\n- **Job title:** UX Designer  \n- **Location:** Remote (based in Portland, Oregon)  \n- **Date of birth:** November\u202f3\u202f1994  \n- **Current salary:** $95,000  \n\n**Career at Insurellm**\n\n| Period | Role | Highlights |\n|--------|------|------------|\n| **January\u202f2022\u202f–\u202fPresent** | UX Designer | • Leads design for the Homellm home‑insurance portal<br>• Conducted user research that lifted user‑satisfaction scores by 35%<br>• Works closely with product and engineering on new feature design |\n| **June\u202f2020\u202f–\u202fDecember\u202f2021** | Junior UX Designer | • Supported senior designers on the Marketllm marketplace redesign<br>• Created wireframes, prototypes, and user‑flow diagrams<br>• Participated in user‑testing and feedback sessions |\n\n**Prior experience**  \n- **August\u202f2018\u202f–\u202fMay\u202f2020:** UI/UX Designer at StartupHub\u202fInc., designing mobile and web interfaces for

In [15]:
view = gr.ChatInterface(fn = answer_question).launch(inbrowser=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/nisumlimbu/Desktop/LLM_Enigneering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/nisumlimbu/Desktop/LLM_Enigneering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/nisumlimbu/Desktop/LLM_Enigneering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2173, in process_api
    inputs = await self.preprocess_data(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/nisumlimbu/Desktop/LLM_Enigneering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1755, in preprocess_data
    self.validate_inputs(block_fn, inputs)
  File "/Users/nisumlimbu/Desktop/LLM_Enigneering/.venv/lib/python3.12/site-packages/gradio/blocks.py",